In [4]:
import torch

# The following is vibe-coded with Gemini

# Use float64 for maximum precision with exponential terms
torch.set_default_dtype(torch.float64)

def calculate_J_with_viable_lambdas(R1, R2, L, U, x_strike, theta):
    """
    Computes J and its gradients while dynamically solving for
    arbitrage-free, viable lambdas at the x_strike junction.

    Layout of theta: [nu1 (R1), sig1_raw (R1), nu2 (R2), sig2_raw (R2)]
    No lambdas are inside theta anymore; they are solved analytically!
    """
    # 1. Slicing the parameters
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1_raw = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2_raw = theta[idx : idx + R2]; idx += R2

    # Enforce strict positivity on sigmas using softplus
    sigs1 = torch.nn.functional.softplus(sigs1_raw) + 1e-6
    sigs2 = torch.nn.functional.softplus(sigs2_raw) + 1e-6

    # =========================================================================
    # STAGE 1: Calculate Unit Base Coefficients (Setting lam1 = 1, lam2 = 1)
    # =========================================================================

    # Left Wing Unit Base
    # c_v1_unit is a vector of two numbers
    # c_v1_unit[0][0] = c^{1, 1}_0
    # c_v1_unit[0][1] = c^{2, 1}_0
    c_v1_unit = []
    dist0 = nus1[0] - L
    s0 = sigs1[0]
    # Boundary at L with lam1_unit = 1.0
    # calculated using distance from L
    c_v1_unit.append((0.5 * (-s0 * 1.0) * torch.exp(-dist0/s0),
                      0.5 * (s0 * 1.0) * torch.exp(dist0/s0)))

    # recursively calculating c^1
    for j in range(R1 - 1):
        P = c_v1_unit[j][0] + c_v1_unit[j][1]
        S = (1.0/sigs1[j]) * (-c_v1_unit[j][0] + c_v1_unit[j][1])
        dist = nus1[j+1] - nus1[j]
        sn = sigs1[j+1]
        c_v1_unit.append((0.5 * (P - sn*S) * torch.exp(-dist/sn),
                          0.5 * (P + sn*S) * torch.exp(dist/sn)))

    # Right Wing Unit Base
    c_v2_unit = [None] * R2
    dist_u = U - nus2[-1]
    sn_u = sigs2[-1]
    # Boundary at U with lam2_unit = 1.0
    c_v2_unit[-1] = (0.5 * (-sn_u * 1.0) * torch.exp(dist_u/sn_u),
                     0.5 * (sn_u * 1.0) * torch.exp(-dist_u/sn_u))

    # recursively calculating c^2
    for j in range(R2 - 1, 0, -1):
        P = c_v2_unit[j][0] + c_v2_unit[j][1]
        S = (1.0/sigs2[j]) * (-c_v2_unit[j][0] + c_v2_unit[j][1])
        dist = nus2[j] - nus2[j-1]
        sp = sigs2[j-1]
        c_v2_unit[j-1] = (0.5 * (P + sp*S) * torch.exp(dist/sp),
                          0.5 * (P - sp*S) * torch.exp(-dist/sp))

    # =========================================================================
    # STAGE 2: Evaluate Unit Wings at Junction x and Solve for Lambdas
    # =========================================================================

    # Left wing unit value and slope at junction strike x
    dist_strike_v1 = x_strike - nus1[-1]
    v1_unit_strike = c_v1_unit[-1][0] * torch.exp(-dist_strike_v1/sigs1[-1]) + \
                    c_v1_unit[-1][1] * torch.exp(dist_strike_v1/sigs1[-1])
    dv1_unit_strike = (1.0/sigs1[-1]) * (-c_v1_unit[-1][0] * torch.exp(-dist_strike_v1/sigs1[-1]) + \
                                          c_v1_unit[-1][1] * torch.exp(dist_strike_v1/sigs1[-1]))

    # Right wing unit value and slope at junction strike x
    dist_strike_v2 = nus2[0] - x_strike
    v2_unit_strike = c_v2_unit[0][0] * torch.exp(dist_strike_v2/sigs2[0]) + \
                    c_v2_unit[0][1] * torch.exp(-dist_strike_v2/sigs2[0])
    dv2_unit_strike = (1.0/sigs2[0]) * (c_v2_unit[0][0] * torch.exp(dist_strike_v2/sigs2[0]) - \
                                        c_v2_unit[0][1] * torch.exp(-dist_strike_v2/sigs2[0]))

    # The denominator tracking the cross-interaction of values and derivatives
    # solve the system of linear equations
    denom = v2_unit_strike * dv1_unit_strike - v1_unit_strike * dv2_unit_strike

    # The unique lambda pair required by Lemma 13 to enforce the derivative gap
    lam1 = v2_unit_strike / denom
    lam2 = v1_unit_strike / denom

    # =========================================================================
    # STAGE 3: Scale Unit Coefficients to Final Viable Coefficients
    # =========================================================================
    c_v1 = [(c1 * lam1, c2 * lam1) for (c1, c2) in c_v1_unit]
    c_v2 = [(c1 * lam2, c2 * lam2) for (c1, c2) in c_v2_unit]

    # =========================================================================
    # STAGE 4: Calculate Smoothness Jumps Using Viable Slopes
    # =========================================================================
    jumps = []

    # Internal Left Wing Jumps
    for j in range(R1 - 1):
        v_sec_left = (1.0/sigs1[j]**2) * (c_v1[j][0] + c_v1[j][1])
        dist = nus1[j+1] - nus1[j]
        v_sec_right = (1.0/sigs1[j+1]**2) * (c_v1[j+1][0]*torch.exp(dist/sigs1[j+1]) +
                                             c_v1[j+1][1]*torch.exp(-dist/sigs1[j+1]))
        jumps.append(v_sec_right - v_sec_left)

    # Junction Jump at x_strike
    v_sec_v1_strike = (1.0/sigs1[-1]**2) * (c_v1[-1][0]*torch.exp(-dist_strike_v1/sigs1[-1]) +
                                            c_v1[-1][1]*torch.exp(dist_strike_v1/sigs1[-1]))
    v_sec_v2_strike = (1.0/sigs2[0]**2) * (c_v2[0][0]*torch.exp(dist_strike_v2/sigs2[0]) +
                                            c_v2[0][1]*torch.exp(-dist_strike_v2/sigs2[0]))
    jumps.append(v_sec_v2_strike - v_sec_v1_strike)

    # Internal Right Wing Jumps
    for j in range(R2 - 1):
        v_sec_left = (1.0/sigs2[j]**2) * (c_v2[j][0] + c_v2[j][1])
        dist = nus2[j+1] - nus2[j]
        v_sec_right = (1.0/sigs2[j+1]**2) * (c_v2[j+1][0]*torch.exp(dist/sigs2[j+1]) +
                                             c_v2[j+1][1]*torch.exp(-dist/sigs2[j+1]))
        jumps.append(v_sec_right - v_sec_left)

    j_val = torch.sum(torch.stack(jumps) ** 2)
    return j_val, lam1, lam2


# --- Verification execution ---
if __name__ == "__main__":
    R1, R2 = 4, 3
    L, U, x_strike = 800.0, 2200.0, 1271.87

    # Initialize unconstrained parameters
    initial_sigs1 = torch.tensor([149.0, 130.0, 250.0, 160.0])
    initial_sigs2 = torch.tensor([155.0, 180.0, 210.0])
    sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1.0)
    sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1.0)

    theta = torch.cat([
        torch.linspace(900, 1200, R1),   # nus1
        sigs1_raw,                       # sigs1_raw
        torch.linspace(1400, 2000, R2),  # nus2
        sigs2_raw                        # sigs2_raw
    ]).clone().detach().requires_grad_(True)

    # Compute J and solve for true independent parameters matching your slides
    # (Notice: S0 and r have been removed completely)
    j_score, true_lam1, true_lam2 = calculate_J_with_viable_lambdas(
        R1, R2, L, U, x_strike, theta
    )

    # Run backward pass to confirm gradients track through the linear scaling system
    j_score.backward()

    print("=== True Slide-Aligned Baseline Simulation ===")
    print(f"Calculated Lambda 1 (Left Wing Scaling):  {true_lam1.item():.6f}")
    print(f"Calculated Lambda 2 (Right Wing Scaling): {true_lam2.item():.6f}")
    print(f"Smoothness Penalty J:                    {j_score.item():.12f}")
    print(f"Gradient Tensor Shape:                   {theta.grad.shape[0]}")
    print("\nGradients (dJ/dtheta) successfully computed:")

    # Optional: Pretty print the gradient tensor components
    labels = []
    for i in range(R1): labels.append(f"nu1_{i}")
    for i in range(R1): labels.append(f"sig1_raw_{i}")
    for i in range(R2): labels.append(f"nu2_{i}")
    for i in range(R2): labels.append(f"sig2_raw_{i}")

    for idx, label in enumerate(labels):
        print(f"  {label:<12}: {theta.grad[idx].item():.4e}")

=== True Slide-Aligned Baseline Simulation ===
Calculated Lambda 1 (Left Wing Scaling):  -1.622490
Calculated Lambda 2 (Right Wing Scaling): 4.144758
Smoothness Penalty J:                    0.024708285324
Gradient Tensor Shape:                   14

Gradients (dJ/dtheta) successfully computed:
  nu1_0       : -8.5051e-06
  nu1_1       : 1.8132e-04
  nu1_2       : -3.5116e-04
  nu1_3       : -1.0842e-19
  sig1_raw_0  : -1.1114e-05
  sig1_raw_1  : -1.8972e-04
  sig1_raw_2  : -1.0714e-04
  sig1_raw_3  : -4.2101e-03
  nu2_0       : -1.4242e-04
  nu2_1       : -2.9592e-04
  nu2_2       : 4.2912e-04
  sig2_raw_0  : 4.8375e-03
  sig2_raw_1  : -1.9278e-04
  sig2_raw_2  : -4.0920e-04


In [7]:
if __name__ == "__main__":
    R1, R2 = 4, 3
    L, U, x_strike = 800.0, 2200.0, 1271.87
    S0, r = 1300.0, 0.043

    # 1. Initialize baseline parameters
    initial_sigs1 = torch.tensor([149.0, 130.0, 250.0, 160.0])
    initial_sigs2 = torch.tensor([155.0, 180.0, 210.0])
    sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1.0)
    sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1.0)

    theta_old = torch.cat([
        torch.linspace(900, 1200, R1),   # nus1
        sigs1_raw,                       # sigs1_raw
        torch.linspace(1400, 2000, R2),  # nus2
        sigs2_raw                        # sigs2_raw
    ]).clone().detach().requires_grad_(True)

    # 2. Compute current smoothness and extract initial gradients
    J_old, _, _ = calculate_J_with_viable_lambdas(
        R1, R2, L, U, x_strike, theta_old
    )
    J_old.backward()
    old_gradients = theta_old.grad.clone().detach()

    print("=== Baseline Evaluation ===")
    print(f"Current Smoothness J_old: {J_old.item():.12f}\n")

    # 3. Perform the Line Search to find a better vector
    learning_rates_to_test = [10.0, 1.0, 0.1, 0.01, 0.001]
    better_theta_found = False
    theta_new_values = None
    alpha_used = None

    for alpha in learning_rates_to_test:
        with torch.no_grad():
            candidate_theta = theta_old - alpha * old_gradients
            J_candidate, _, _ = calculate_J_with_viable_lambdas(
                R1, R2, L, U, x_strike, candidate_theta
            )

            if J_candidate.item() < J_old.item():
                theta_new_values = candidate_theta.clone().detach()
                alpha_used = alpha
                better_theta_found = True
                break

    # 4. If a better vector exists, compute its gradients explicitly
    if better_theta_found:
        # Create a fresh tensor tracking gradients for the new position
        theta_new = theta_new_values.clone().detach().requires_grad_(True)

        # Run a fresh forward pass to calculate the new gradient graph
        J_new, _, _ = calculate_J_with_viable_lambdas(
            R1, R2, L, U, x_strike, theta_new
        )
        J_new.backward()
        new_gradients = theta_new.grad.clone().detach()

        print(f"=== Step Successful (α = {alpha_used}) ===")
        print(f"Improved Smoothness J_new: {J_new.item():.12f}")
        print(f"Total Penalty Reduction: {((J_old - J_new)/J_old * 100).item():.4f}%\n")

        # 5. Build Parameter Label Mapping for clear printing
        labels = []
        for i in range(R1): labels.append(f"nu1_{i}")
        for i in range(R1): labels.append(f"sig1_raw_{i}")
        for i in range(R2): labels.append(f"nu2_{i}")
        for i in range(R2): labels.append(f"sig2_raw_{i}")

        # 6. Print Comparison Table
        print(f"{'Parameter':<12} | {'Old Value':<12} | {'Old Gradient':<15} | {'New Value':<12} | {'New Gradient':<15}")
        print("-" * 76)

        for idx in range(len(labels)):
            print(f"{labels[idx]:<12} | "
                  f"{theta_old[idx].item():<12.4f} | "
                  f"{old_gradients[idx].item():<15.4e} | "
                  f"{theta_new[idx].item():<12.4f} | "
                  f"{new_gradients[idx].item():<15.4e}")

    else:
        print("Could not find an immediate improvement using basic gradient steps.")

=== Baseline Evaluation ===
Current Smoothness J_old: 0.024708285324

=== Step Successful (α = 10.0) ===
Improved Smoothness J_new: 0.024295107175
Total Penalty Reduction: 1.6722%

Parameter    | Old Value    | Old Gradient    | New Value    | New Gradient   
----------------------------------------------------------------------------
nu1_0        | 900.0000     | -8.5051e-06     | 900.0001     | -8.2954e-06    
nu1_1        | 1000.0000    | 1.8132e-04      | 999.9982     | 1.7694e-04     
nu1_2        | 1100.0000    | -3.5116e-04     | 1100.0035    | -3.4229e-04    
nu1_3        | 1200.0000    | -1.0842e-19     | 1200.0000    | -2.1684e-19    
sig1_raw_0   | 149.0000     | -1.1114e-05     | 149.0001     | -1.0838e-05    
sig1_raw_1   | 130.0000     | -1.8972e-04     | 130.0019     | -1.8522e-04    
sig1_raw_2   | 250.0000     | -1.0714e-04     | 250.0011     | -1.0448e-04    
sig1_raw_3   | 160.0000     | -4.2101e-03     | 160.0421     | -4.1035e-03    
nu2_0        | 1400.0000    | -